# 爆量篩選
篩出當天預估成交金額相對 20 日均量明顯放大的股票

**公式（每分鐘計算）：**
- 預估成交金額 = 當分鐘最高價 × 預估全日總量 × 1000（vol_exp 單位為張，1張=1000股）
- 爆量倍數 = 預估成交金額 / avg_turnover_20d

In [2]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ===== 參數設定 =====
TARGET_DATE = '2025-12-31'          # 篩選日期
THRESHOLD = 2.0                     # 爆量門檻 (倍數)
THRESHOLD_LARGE = 1.2               # 大型股爆量門檻 (倍數)
LARGE_STOCK_TURNOVER = 5e9          # 大型股定義: 20日均量 > 50億
MIN_EST_TURNOVER = 5e8              # 預估成交金額門檻 (5億)

# ===== 路徑 =====
VOL_EXP_DIR = r'C:\Users\user\Documents\_15_預估量\exp_vol_project\tw_kbar_1m_vol_exp'
KBAR_1M_DIR = r'C:\Users\user\Documents\_04_lupFinmind\1min'
STOCK_VOL_PATH = r'C:\Users\user\Documents\_15_預估量\exp_vol_project\data\stock_volume.parquet'

In [3]:
# ===== 1. 載入 stock_volume =====
df_stock_vol = pd.read_parquet(STOCK_VOL_PATH)
df_stock_vol['date_str'] = df_stock_vol['date'].astype(str).str[:10]

# 當日資料
df_daily = df_stock_vol[df_stock_vol['date_str'] == TARGET_DATE][['stock_id', 'avg_turnover_20d', 'is_prev_limit_up']]
df_daily = df_daily.set_index('stock_id')

# 前一交易日收盤價 (計算漲停價用)
all_dates = sorted(df_stock_vol['date_str'].unique())
prev_date = all_dates[all_dates.index(TARGET_DATE) - 1]
df_prev = df_stock_vol[df_stock_vol['date_str'] == prev_date][['stock_id', 'close']].set_index('stock_id')

# 計算漲停價
def calc_limit_up(prev_close):
    res = prev_close * 1.1
    tick = pd.Series(0.01, index=res.index)
    tick[res >= 10] = 0.05
    tick[res >= 50] = 0.1
    tick[res >= 100] = 0.5
    tick[res >= 500] = 1
    tick[res >= 1000] = 5
    return (np.floor((res + 0.0001) / tick) * tick).round(2)

limit_up_prices = calc_limit_up(df_prev['close'])
limit_up_prices.name = 'limit_up'

print(f'stock_volume: {TARGET_DATE} 共 {len(df_daily)} 檔, 前一交易日: {prev_date}')

# ===== 2. 載入 vol_exp =====
date_compact = TARGET_DATE.replace('-', '')
df_vol_exp = pd.read_parquet(os.path.join(VOL_EXP_DIR, f'vol_exp_{date_compact}.parquet'))
print(f'vol_exp: {df_vol_exp.shape[0]} 分鐘 x {df_vol_exp.shape[1]} 檔股票')

stock_volume: 2025-12-31 共 1907 檔, 前一交易日: 2025-12-30
vol_exp: 266 分鐘 x 477 檔股票


In [4]:
# ===== 3. 批次載入 1min K (H, C) =====
stock_codes = df_vol_exp.columns.tolist()
kbar_date_dir = os.path.join(KBAR_1M_DIR, TARGET_DATE)

data_high, data_close = {}, {}
missing = []

for code in stock_codes:
    fpath = os.path.join(kbar_date_dir, f'{code}.parquet')
    if not os.path.exists(fpath):
        missing.append(code)
        continue
    df_k = pd.read_parquet(fpath, columns=['minute', 'high', 'close'])
    df_k['datetime'] = pd.to_datetime(f'{TARGET_DATE} ' + df_k['minute'])
    df_k = df_k.set_index('datetime')
    data_high[code] = df_k['high']
    data_close[code] = df_k['close']

df_high = pd.DataFrame(data_high)
df_close = pd.DataFrame(data_close)
print(f'成功載入 {len(data_high)} 檔, {len(missing)} 檔無 1min K 資料')

成功載入 477 檔, 0 檔無 1min K 資料


In [5]:
# ===== 4. 計算每分鐘爆量倍數 =====
common_stocks = df_high.columns.intersection(df_vol_exp.columns).intersection(df_daily.index)
common_times = df_high.index.intersection(df_vol_exp.index)

df_high_a = df_high.loc[common_times, common_stocks]
df_vol_exp_a = df_vol_exp.loc[common_times, common_stocks]
df_close_a = df_close.loc[common_times, common_stocks]
avg_turnover = df_daily.loc[common_stocks, 'avg_turnover_20d']

# 預估成交金額 = 當分鐘最高價 × 預估量 × 1000
df_est_turnover = df_high_a * df_vol_exp_a * 1000
# 爆量倍數
df_ratio = df_est_turnover.div(avg_turnover, axis=1)

print(f'計算完成: {df_ratio.shape[0]} 分鐘 x {df_ratio.shape[1]} 檔股票')

計算完成: 266 分鐘 x 475 檔股票


In [6]:
# ===== 5. 篩選: 爆量 + 預估成交金額>5億 + 非漲停 + 前日非漲停 =====
# 依 20日均量分級門檻: 大型股(>50億) 用 1.2x, 其他用 2.0x
is_large = avg_turnover > LARGE_STOCK_TURNOVER
threshold_per_stock = pd.Series(THRESHOLD, index=common_stocks)
threshold_per_stock[is_large] = THRESHOLD_LARGE

# 每分鐘每檔的門檻矩陣
threshold_matrix = pd.DataFrame(
    np.tile(threshold_per_stock.values, (len(common_times), 1)),
    index=common_times, columns=common_stocks
)

boom_mask = (df_ratio >= threshold_matrix) & (df_est_turnover >= MIN_EST_TURNOVER)

# 漲停遮罩
stocks_with_limit = common_stocks.intersection(limit_up_prices.index)
limit_mask = df_high_a[stocks_with_limit].eq(limit_up_prices[stocks_with_limit], axis=1)

# 有效爆量 = 爆量但不在漲停
boom_at_limit = boom_mask[stocks_with_limit] & limit_mask
valid_boom = boom_mask.copy()
valid_boom[stocks_with_limit] = boom_mask[stocks_with_limit] & ~limit_mask

# 篩出曾有有效爆量的股票
boom_stocks = valid_boom.any(axis=0)
boom_stock_list = boom_stocks[boom_stocks].index.tolist()

# 剔除前日漲停股
prev_limit_stocks = df_daily[df_daily['is_prev_limit_up'] == True].index
removed_prev_limit = [s for s in boom_stock_list if s in prev_limit_stocks]
boom_stock_list = [s for s in boom_stock_list if s not in prev_limit_stocks]

# 統計表格
rows = []
for code in boom_stock_list:
    boom_times = valid_boom.index[valid_boom[code]]
    first_boom = boom_times[0]
    max_ratio = df_ratio.loc[boom_times, code].max()
    max_time = df_ratio.loc[boom_times, code].idxmax()
    max_est = df_est_turnover.loc[max_time, code]
    stock_threshold = threshold_per_stock[code]
    rows.append({
        'stock_id': code,
        '門檻': f'{stock_threshold}x',
        '首次爆量時間': first_boom.strftime('%H:%M'),
        '最大倍數時間': max_time.strftime('%H:%M'),
        '最大爆量倍數': round(max_ratio, 2),
        '最大預估成交金額(億)': round(max_est / 1e8, 1),
        '20日均量(億)': round(avg_turnover[code] / 1e8, 1),
        '爆量分鐘數': len(boom_times),
    })

df_table = pd.DataFrame(rows).sort_values('最大爆量倍數', ascending=False).reset_index(drop=True)
n_large = sum(1 for r in rows if r['門檻'] == f'{THRESHOLD_LARGE}x')
print(f'=== {TARGET_DATE} 爆量篩選 (一般>={THRESHOLD}x, 大型股>={THRESHOLD_LARGE}x, 預估成交金額>=5億, 前日非漲停) ===')
print(f'有效爆量股: {len(df_table)} 檔 (大型股{n_large}檔) | 漲停剔除分鐘: {boom_at_limit.sum().sum():.0f} | 前日漲停剔除: {len(removed_prev_limit)} 檔')
df_table

=== 2025-12-31 爆量篩選 (一般>=2.0x, 大型股>=1.2x, 預估成交金額>=5億, 前日非漲停) ===
有效爆量股: 55 檔 (大型股7檔) | 漲停剔除分鐘: 1403 | 前日漲停剔除: 8 檔


,stock_id,門檻,首次爆量時間,最大倍數時間,最大爆量倍數,最大預估成交金額(億),20日均量(億),爆量分鐘數
0,3049,2.0x,09:00,09:00,33.29,5.4,0.2,2
1,5386,2.0x,09:00,09:01,10.73,11.2,1.0,258
2,8422,2.0x,09:02,13:00,6.80,60.9,9.0,162
3,7610,2.0x,09:00,09:00,6.12,5.5,0.9,1
4,2409,2.0x,09:13,10:18,5.46,35.3,6.5,253
5,8064,2.0x,09:00,09:24,4.91,7.2,1.5,258
6,1528,2.0x,09:02,13:30,4.88,12.4,2.5,264
7,3701,2.0x,09:00,13:30,4.62,13.2,2.9,265
8,8028,2.0x,09:42,12:31,4.22,42.2,10.0,224
9,3264,2.0x,09:03,09:07,4.22,59.8,14.2,60


In [20]:
# ===== 7. 個股查詢: 走勢圖 + 預估成交金額走勢 =====
def plot_stock_detail(stock_id, date):
    """輸入股票代碼與日期，畫出走勢圖 + 預估成交金額走勢"""
    date_compact = date.replace('-', '')

    # 載入 1min K
    kbar_path = os.path.join(KBAR_1M_DIR, date, f'{stock_id}.parquet')
    if not os.path.exists(kbar_path):
        print(f'找不到 {stock_id} 在 {date} 的 1min K 資料')
        return
    df_k = pd.read_parquet(kbar_path)
    df_k['datetime'] = pd.to_datetime(date + ' ' + df_k['minute'])
    df_k = df_k.set_index('datetime')

    # 載入 vol_exp
    vol_path = os.path.join(VOL_EXP_DIR, f'vol_exp_{date_compact}.parquet')
    if not os.path.exists(vol_path):
        print(f'找不到 {date} 的 vol_exp 資料')
        return
    df_ve = pd.read_parquet(vol_path)
    if stock_id not in df_ve.columns:
        print(f'{stock_id} 不在 vol_exp 中')
        return
    vol_series = df_ve[stock_id]

    # 對齊時間
    common_idx = df_k.index.intersection(vol_series.index)
    high = df_k.loc[common_idx, 'high']
    vol_exp = vol_series.loc[common_idx]
    est_turnover = high * vol_exp * 1000  # 預估成交金額(元)

    # 查 avg_turnover_20d
    df_sv = pd.read_parquet(STOCK_VOL_PATH)
    df_sv['date_str'] = df_sv['date'].astype(str).str[:10]
    row = df_sv[(df_sv['date_str'] == date) & (df_sv['stock_id'] == stock_id)]
    avg_20d = row['avg_turnover_20d'].values[0] if not row.empty else None

    # 畫圖
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(f'{stock_id} 走勢圖', f'{stock_id} 預估成交金額'),
        vertical_spacing=0.08, row_heights=[0.5, 0.5]
    )

    # 上圖: 收盤價走勢
    fig.add_trace(go.Scatter(
        x=common_idx, y=df_k.loc[common_idx, 'close'].values,
        mode='lines', name='收盤價',
        line=dict(width=1.5, color='#1f77b4')
    ), row=1, col=1)

    # 下圖: 預估成交金額
    fig.add_trace(go.Scatter(
        x=common_idx, y=est_turnover.values / 1e8,
        mode='lines', name='預估成交金額(億)',
        line=dict(width=1.5, color='#ff7f0e')
    ), row=2, col=1)

    # 20日均量參考線
    if avg_20d is not None:
        fig.add_hline(
            y=avg_20d / 1e8, row=2, col=1,
            line=dict(color='white', width=1, dash='dash'),
            annotation_text=f'20日均量 {avg_20d/1e8:.1f}億',
            annotation_position='top left', annotation_font_size=10
        )

    fig.update_layout(
        height=700,
        title_text=f'{stock_id} - {date} 日內分析',
        template='plotly_dark', showlegend=False
    )
    fig.update_xaxes(tickformat='%H:%M')
    fig.update_yaxes(title_text='價格', row=1, col=1)
    fig.update_yaxes(title_text='預估成交金額(億)', row=2, col=1)
    fig.show()

# === 使用方式 ===
plot_stock_detail('2337', '2026-02-04')

In [8]:
# ===== 8. 批量篩選 2025-09-01 至今，輸出 parquet =====
OUTPUT_PATH = r'C:\Users\user\Documents\_15_預估量\exp_vol_project\data\boom_stocks_since_20250901.parquet'
START_DATE = '2025-09-01'

# 載入 stock_volume (一次性)
df_stock_vol_all = pd.read_parquet(STOCK_VOL_PATH)
df_stock_vol_all['date_str'] = df_stock_vol_all['date'].astype(str).str[:10]
all_dates_sorted = sorted(df_stock_vol_all['date_str'].unique())
target_dates = [d for d in all_dates_sorted if d >= START_DATE]

print(f'掃描範圍: {target_dates[0]} ~ {target_dates[-1]}, 共 {len(target_dates)} 個交易日')

def calc_limit_up_batch(prev_close):
    """計算漲停價"""
    res = prev_close * 1.1
    tick = pd.Series(0.01, index=res.index)
    tick[res >= 10] = 0.05
    tick[res >= 50] = 0.1
    tick[res >= 100] = 0.5
    tick[res >= 500] = 1
    tick[res >= 1000] = 5
    return (np.floor((res + 0.0001) / tick) * tick).round(2)

all_results = []
skipped = []

for idx, date in enumerate(target_dates):
    date_compact = date.replace('-', '')

    # 檢查檔案是否存在
    vol_exp_path = os.path.join(VOL_EXP_DIR, f'vol_exp_{date_compact}.parquet')
    kbar_date_dir = os.path.join(KBAR_1M_DIR, date)
    if not os.path.exists(vol_exp_path) or not os.path.exists(kbar_date_dir):
        skipped.append(date)
        continue

    # 當日 stock_volume
    df_daily_d = df_stock_vol_all[df_stock_vol_all['date_str'] == date][['stock_id', 'avg_turnover_20d', 'is_prev_limit_up']]
    df_daily_d = df_daily_d.set_index('stock_id')
    if df_daily_d.empty:
        skipped.append(date)
        continue

    # 前一交易日 -> 漲停價
    date_idx = all_dates_sorted.index(date)
    if date_idx == 0:
        skipped.append(date)
        continue
    prev_date_d = all_dates_sorted[date_idx - 1]
    df_prev_d = df_stock_vol_all[df_stock_vol_all['date_str'] == prev_date_d][['stock_id', 'close']].set_index('stock_id')
    limit_up_d = calc_limit_up_batch(df_prev_d['close'])

    # 載入 vol_exp
    df_vol_exp_d = pd.read_parquet(vol_exp_path)

    # 載入 1min K (high)
    stock_codes_d = df_vol_exp_d.columns.tolist()
    data_high_d = {}
    for code in stock_codes_d:
        fpath = os.path.join(kbar_date_dir, f'{code}.parquet')
        if not os.path.exists(fpath):
            continue
        df_k = pd.read_parquet(fpath, columns=['minute', 'high'])
        df_k['datetime'] = pd.to_datetime(f'{date} ' + df_k['minute'])
        df_k = df_k.set_index('datetime')
        data_high_d[code] = df_k['high']

    if not data_high_d:
        skipped.append(date)
        continue

    df_high_d = pd.DataFrame(data_high_d)

    # 計算爆量
    common_stocks_d = df_high_d.columns.intersection(df_vol_exp_d.columns).intersection(df_daily_d.index)
    common_times_d = df_high_d.index.intersection(df_vol_exp_d.index)
    if len(common_stocks_d) == 0 or len(common_times_d) == 0:
        skipped.append(date)
        continue

    df_high_ad = df_high_d.loc[common_times_d, common_stocks_d]
    df_vol_exp_ad = df_vol_exp_d.loc[common_times_d, common_stocks_d]
    avg_turnover_d = df_daily_d.loc[common_stocks_d, 'avg_turnover_20d']

    df_est_turnover_d = df_high_ad * df_vol_exp_ad * 1000
    df_ratio_d = df_est_turnover_d.div(avg_turnover_d, axis=1)

    # 門檻
    is_large_d = avg_turnover_d > LARGE_STOCK_TURNOVER
    threshold_d = pd.Series(THRESHOLD, index=common_stocks_d)
    threshold_d[is_large_d] = THRESHOLD_LARGE
    threshold_mat_d = pd.DataFrame(
        np.tile(threshold_d.values, (len(common_times_d), 1)),
        index=common_times_d, columns=common_stocks_d
    )

    # 爆量遮罩 (不含漲停過濾，僅量能條件)
    boom_mask_d = (df_ratio_d >= threshold_mat_d) & (df_est_turnover_d >= MIN_EST_TURNOVER)
    boom_stocks_d = boom_mask_d.any(axis=0)
    boom_list_d = boom_stocks_d[boom_stocks_d].index.tolist()

    # 逐檔收集結果
    for code in boom_list_d:
        boom_times_d = boom_mask_d.index[boom_mask_d[code]]
        first_boom_time = boom_times_d[0]

        # is_limitup_prev: 前日是否漲停
        is_lup_prev = bool(df_daily_d.loc[code, 'is_prev_limit_up'])

        # is_limitup: 當日是否曾觸及漲停價
        is_lup = False
        if code in limit_up_d.index:
            lup_price = limit_up_d[code]
            is_lup = bool((df_high_ad[code] == lup_price).any())

        all_results.append({
            'date': date,
            'stock_id': code,
            '首次爆量時間': first_boom_time.strftime('%H:%M'),
            'is_limitup_prev': is_lup_prev,
            'is_limitup': is_lup,
        })

    if (idx + 1) % 20 == 0 or idx == len(target_dates) - 1:
        print(f'  進度: {idx+1}/{len(target_dates)} ({date}), 累計 {len(all_results)} 筆')

# 輸出
df_boom_all = pd.DataFrame(all_results)
df_boom_all.to_parquet(OUTPUT_PATH, index=False)

print(f'\n===== 完成 =====')
print(f'有效交易日: {len(target_dates) - len(skipped)} 日, 跳過: {len(skipped)} 日')
print(f'總筆數: {len(df_boom_all)}')
print(f'儲存路徑: {OUTPUT_PATH}')
print(f'\n各欄位說明:')
print(f'  date         - 交易日期')
print(f'  stock_id     - 股票代碼')
print(f'  首次爆量時間  - 當日首次符合爆量條件的時間 (HH:MM)')
print(f'  is_limitup_prev - 前一交易日是否漲停')
print(f'  is_limitup      - 當日是否曾觸及漲停價')
print(f'\n前 20 筆預覽:')
df_boom_all.head(20)

掃描範圍: 2025-09-01 ~ 2026-02-05, 共 108 個交易日
  進度: 20/108 (2025-09-26), 累計 1184 筆
  進度: 40/108 (2025-10-30), 累計 2251 筆
  進度: 60/108 (2025-11-27), 累計 3207 筆
  進度: 80/108 (2025-12-26), 累計 4263 筆
  進度: 100/108 (2026-01-26), 累計 6076 筆
  進度: 108/108 (2026-02-05), 累計 6700 筆

===== 完成 =====
有效交易日: 108 日, 跳過: 0 日
總筆數: 6700
儲存路徑: C:\Users\user\Documents\_15_預估量\exp_vol_project\data\boom_stocks_since_20250901.parquet

各欄位說明:
  date         - 交易日期
  stock_id     - 股票代碼
  首次爆量時間  - 當日首次符合爆量條件的時間 (HH:MM)
  is_limitup_prev - 前一交易日是否漲停
  is_limitup      - 當日是否曾觸及漲停價

前 20 筆預覽:


,date,stock_id,首次爆量時間,is_limitup_prev,is_limitup
0,2025-09-01,1503,09:00,False,False
1,2025-09-01,1785,10:08,False,False
2,2025-09-01,2208,09:00,False,False
3,2025-09-01,2347,09:00,False,False
4,2025-09-01,2355,11:08,False,False
5,2025-09-01,2365,10:33,False,False
6,2025-09-01,2368,11:43,False,False
7,2025-09-01,2401,09:22,False,False
8,2025-09-01,2449,09:00,True,False
9,2025-09-01,2464,09:00,False,False


In [9]:
# ===== 9-1. 首次爆量時間分布 (長條圖) =====
# 若尚未執行 cell 8，可從 parquet 讀取:
# df_boom_all = pd.read_parquet(OUTPUT_PATH)

# 以 30 分鐘為一組做分箱
time_minutes = pd.to_timedelta(df_boom_all['首次爆量時間'].apply(lambda t: t + ':00'))
bin_labels = []
bin_counts = []
bins = [
    ('09:00', '09:30'), ('09:30', '10:00'), ('10:00', '10:30'), ('10:30', '11:00'),
    ('11:00', '11:30'), ('11:30', '12:00'), ('12:00', '12:30'), ('12:30', '13:00'),
    ('13:00', '13:30'),
]
for start, end in bins:
    s = pd.to_timedelta(start + ':00')
    e = pd.to_timedelta(end + ':00')
    if end == '13:30':
        mask = (time_minutes >= s) & (time_minutes <= e)
    else:
        mask = (time_minutes >= s) & (time_minutes < e)
    bin_labels.append(f'{start}-{end}')
    bin_counts.append(mask.sum())

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=bin_labels,
    y=bin_counts,
    text=bin_counts,
    textposition='outside',
    marker_color='#636EFA',
))
fig1.update_layout(
    title='首次爆量時間分布 (2025-09-01 起)',
    xaxis_title='首次爆量時間區間',
    yaxis_title='出現次數',
    template='plotly_dark',
    height=450,
)
fig1.show()

print(f'總筆數: {sum(bin_counts)}')
for lbl, cnt in zip(bin_labels, bin_counts):
    print(f'  {lbl}: {cnt} 筆 ({cnt/sum(bin_counts)*100:.1f}%)')

總筆數: 6700
  09:00-09:30: 4911 筆 (73.3%)
  09:30-10:00: 441 筆 (6.6%)
  10:00-10:30: 267 筆 (4.0%)
  10:30-11:00: 201 筆 (3.0%)
  11:00-11:30: 154 筆 (2.3%)
  11:30-12:00: 122 筆 (1.8%)
  12:00-12:30: 120 筆 (1.8%)
  12:30-13:00: 138 筆 (2.1%)
  13:00-13:30: 346 筆 (5.2%)


In [10]:
# ===== 9-2. 每日爆量標的數量折線圖 =====
daily_counts = df_boom_all.groupby('date').size().reset_index(name='count')

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=daily_counts['date'],
    y=daily_counts['count'],
    mode='lines+markers',
    marker=dict(size=5, color='#EF553B'),
    line=dict(width=1.5, color='#EF553B'),
    text=daily_counts['count'],
    hovertemplate='%{x}<br>爆量標的: %{y} 檔<extra></extra>',
))

# 平均線
avg_count = daily_counts['count'].mean()
fig2.add_hline(
    y=avg_count,
    line=dict(color='white', width=1, dash='dash'),
    annotation_text=f'平均 {avg_count:.1f} 檔',
    annotation_position='top left',
    annotation_font_size=11,
)

fig2.update_layout(
    title='每日爆量標的數量 (2025-09-01 起)',
    xaxis_title='日期',
    yaxis_title='爆量標的數',
    template='plotly_dark',
    height=450,
    xaxis=dict(tickformat='%Y-%m-%d', tickangle=-45),
)
fig2.show()

print(f'期間: {daily_counts["date"].iloc[0]} ~ {daily_counts["date"].iloc[-1]}')
print(f'交易日數: {len(daily_counts)}')
print(f'每日平均: {avg_count:.1f} 檔')
print(f'最多: {daily_counts.loc[daily_counts["count"].idxmax(), "date"]} ({daily_counts["count"].max()} 檔)')
print(f'最少: {daily_counts.loc[daily_counts["count"].idxmin(), "date"]} ({daily_counts["count"].min()} 檔)')

期間: 2025-09-01 ~ 2026-02-05
交易日數: 108
每日平均: 62.0 檔
最多: 2026-01-29 (147 檔)
最少: 2025-11-19 (25 檔)
